# Cuaderno de Clase — Síntesis de Representaciones Semánticas

## Objetivo

Integrar y comparar las principales técnicas de representación semántica de texto que vimos en esta unidad: Bag-of-Words, TF-IDF y embeddings de palabras (Word2Vec/FastText). La idea es entender cuándo conviene usar cada una y qué información captura cada enfoque.

## Resultados de aprendizaje

Al terminar este cuaderno vas a poder:

- Contrastar en la práctica las diferencias entre BoW, TF-IDF y embeddings.
- Usar n-gramas en TF-IDF para capturar información de contexto local.
- Cargar un modelo de embeddings y explorar similitudes con él.
- Elegir fundamentadamente una representación para una tarea dada.

## Relación con cuadernos anteriores

Este cuaderno integra lo trabajado en toda la unidad: BoW y TF-IDF (módulo de vectorización), Word2Vec y FastText (primeros dos cuadernos de esta parte), y sentence embeddings (cuadernos 3 y 4). Acá los vemos uno al lado del otro para poder compararlos directamente.


## Microglosario

Antes de empezar, repasamos los términos centrales de esta clase:

| Término | Definición breve |
|---|---|
| **BoW** (Bag-of-Words) | Representa un texto contando cuántas veces aparece cada palabra. Ignora el orden. |
| **TF-IDF** | Ajusta las cuentas de BoW para que las palabras muy frecuentes en todo el corpus pesen menos. |
| **N-grama** | Secuencia de N palabras consecutivas tratada como unidad. "mate amargo" es un bigrama. |
| **Embedding** | Vector denso de baja dimensión que captura significado semántico. |
| **Pipeline** | Cadena de pasos (vectorización + modelo) que se ejecutan en orden. |
| **Vocabulario** | Conjunto de palabras únicas que conoce un modelo dado. |


## 1. Importaciones

Cargamos las herramientas que vamos a usar a lo largo del cuaderno.


In [1]:
# Operaciones numéricas
import numpy as np

# Vectorización: BoW y TF-IDF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Para trabajar con vectores de palabras pre-entrenados
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente.")


Librerías importadas correctamente.


## 2. Corpus de ejemplo

Vamos a trabajar con un corpus pequeño de textos en español sobre un mismo tema (el mate, para mantener el contexto cultural). Los corpus pequeños nos permiten inspeccionare los resultados completos sin que se corte la pantalla.

> **Para mirar:** ¿cuántas palabras únicas tiene este corpus? ¿Hay palabras que aparecen en todos los textos? ¿Qué pasa con esas palabras en TF-IDF?


In [2]:
# Corpus de ejemplo: textos cortos en español
corpus = [
    "qué bueno está el mate amargo",
    "el mate dulce no me va",
    "qué amargo está este mate che",
    "me gusta el mate bien caliente"
]

# Mostramos los textos numerados para referencia
print("Corpus de trabajo:")
for i in range(len(corpus)):
    numero = i + 1
    texto = corpus[i]
    print(f"  [{numero}] {texto}")


Corpus de trabajo:
  [1] qué bueno está el mate amargo
  [2] el mate dulce no me va
  [3] qué amargo está este mate che
  [4] me gusta el mate bien caliente


15 palabras. Mate   aparece en todos

## 3. Representación BoW (Bag-of-Words)

BoW convierte cada texto en un vector de cuentas: cuántas veces aparece cada palabra del vocabulario en ese texto. El orden de las palabras se pierde completamente.

> **Para mirar:** ¿qué palabras tienen un conteo mayor en el vector BoW? ¿Son las palabras más "importantes" o las más comunes?


In [4]:
# Creamos el vectorizador de Bag-of-Words
# 'CountVectorizer' cuenta cuántas veces aparece cada palabra en cada documento
contador_bow = CountVectorizer()

# 'fit_transform' aprende el vocabulario y convierte los textos en una sola operación
# Primero mostramos cómo separarlo en dos pasos para entender qué hace cada uno:

# Paso 1: aprender el vocabulario del corpus
contador_bow.fit(corpus)

# Paso 2: convertir los textos a vectores de cuentas
matriz_bow = contador_bow.transform(corpus)

# Mostramos el vocabulario aprendido
vocabulario_bow = contador_bow.get_feature_names_out()
print("Vocabulario del corpus (ordenado alfabéticamente):")
print(vocabulario_bow)
print()

# Mostramos la matriz completa como un array denso
# '.toarray()' convierte el formato esparso a un array que podemos leer
print("Matriz BoW (filas = documentos, columnas = palabras):")
print(matriz_bow.toarray())


Vocabulario del corpus (ordenado alfabéticamente):
['amargo' 'bien' 'bueno' 'caliente' 'che' 'dulce' 'el' 'este' 'está'
 'gusta' 'mate' 'me' 'no' 'qué' 'va']

Matriz BoW (filas = documentos, columnas = palabras):
[[1 0 1 0 0 0 1 0 1 0 1 0 0 1 0]
 [0 0 0 0 0 1 1 0 0 0 1 1 1 0 1]
 [1 0 0 0 1 0 0 1 1 0 1 0 0 1 0]
 [0 1 0 1 0 0 1 0 0 1 1 1 0 0 0]]


Las palabras con conteo mayor son las que aoarecen mas dentro del documento. Este corpus como mate tiene 1 en todas las filas porque es masfrecuente.

## 4. Representación TF-IDF

TF-IDF ajusta las cuentas de BoW: las palabras que aparecen en muchos documentos (como "el", "qué", "me") reciben un peso más bajo, mientras que las palabras que son distintivas de un documento particular reciben un peso más alto.

> **Para mirar:** compará la matriz BoW con la de TF-IDF. ¿Qué palabras bajan de peso en TF-IDF? ¿Qué palabras suben?


In [5]:
# Creamos el vectorizador TF-IDF
vectorizador_tfidf = TfidfVectorizer()

# Aprendemos el vocabulario y transformamos en un solo paso
# (el vocabulario es el mismo que en BoW si no cambiamos parámetros)
vectorizador_tfidf.fit(corpus)
matriz_tfidf = vectorizador_tfidf.transform(corpus)

# Mostramos la matriz TF-IDF redondeada a 3 decimales para facilitar la lectura
import numpy as np
print("Matriz TF-IDF (filas = documentos, columnas = palabras):")
print(matriz_tfidf.toarray().round(3))
print()
print("Observación: los valores son ahora decimales, no enteros.")
print("Las palabras muy comunes en todo el corpus tienen valores más bajos.")


Matriz TF-IDF (filas = documentos, columnas = palabras):
[[0.419 0.    0.531 0.    0.    0.    0.339 0.    0.419 0.    0.277 0.
  0.    0.419 0.   ]
 [0.    0.    0.    0.    0.    0.482 0.308 0.    0.    0.    0.252 0.38
  0.482 0.    0.482]
 [0.388 0.    0.    0.    0.492 0.    0.    0.492 0.388 0.    0.257 0.
  0.    0.388 0.   ]
 [0.    0.482 0.    0.482 0.    0.    0.308 0.    0.    0.482 0.252 0.38
  0.    0.    0.   ]]

Observación: los valores son ahora decimales, no enteros.
Las palabras muy comunes en todo el corpus tienen valores más bajos.


La palabra mate tiene el peso mas bajo

## 5. TF-IDF con N-gramas

Hasta ahora solo usamos palabras sueltas (unigramas). Pero algunas ideas solo tienen sentido como secuencia: "mate amargo" no es lo mismo que "amargo" + "mate" por separado.

Al agregar bigramas al vectorizador, el vocabulario crece y capturamos algo de contexto local.

> **Para mirar:** ¿qué bigramas aparecen en el vocabulario? ¿Te parecen útiles para distinguir los documentos?


In [6]:
# Creamos un vectorizador TF-IDF que incluye unigramas y bigramas
# 'ngram_range=(1, 2)' significa: incluir palabras solas (1) y pares de palabras (2)
vectorizador_ngram = TfidfVectorizer(ngram_range=(1, 2))

# Aprendemos el vocabulario extendido
vectorizador_ngram.fit(corpus)
matriz_ngram = vectorizador_ngram.transform(corpus)

# El vocabulario ahora incluye tanto palabras como pares
vocabulario_ngram = vectorizador_ngram.get_feature_names_out()
print(f"Vocabulario con unigramas y bigramas ({len(vocabulario_ngram)} términos):")
print(vocabulario_ngram)
print()

# Mostramos la matriz
print("Matriz TF-IDF con N-gramas:")
print(matriz_ngram.toarray().round(3))


Vocabulario con unigramas y bigramas (33 términos):
['amargo' 'amargo está' 'bien' 'bien caliente' 'bueno' 'bueno está'
 'caliente' 'che' 'dulce' 'dulce no' 'el' 'el mate' 'este' 'este mate'
 'está' 'está el' 'está este' 'gusta' 'gusta el' 'mate' 'mate amargo'
 'mate bien' 'mate che' 'mate dulce' 'me' 'me gusta' 'me va' 'no' 'no me'
 'qué' 'qué amargo' 'qué bueno' 'va']

Matriz TF-IDF con N-gramas:
[[0.28  0.    0.    0.    0.355 0.355 0.    0.    0.    0.    0.226 0.226
  0.    0.    0.28  0.355 0.    0.    0.    0.185 0.355 0.    0.    0.
  0.    0.    0.    0.    0.    0.28  0.    0.355 0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.339 0.339 0.216 0.216
  0.    0.    0.    0.    0.    0.    0.    0.177 0.    0.    0.    0.339
  0.267 0.    0.339 0.339 0.339 0.    0.    0.    0.339]
 [0.261 0.331 0.    0.    0.    0.    0.    0.331 0.    0.    0.    0.
  0.331 0.331 0.261 0.    0.331 0.    0.    0.173 0.    0.    0.331 0.
  0.    0.    0.    0.    0.    0.261 0.331 0.   

Si mate amargo distinto de mate dulce. Solo la desventaja seria que crecio en dimensionalidad

## 6. Embeddings de palabras pre-entrenados

Ahora vamos a cargar un modelo de embeddings para comparar directamente cómo representa el significado en comparación con BoW y TF-IDF.

Para esto usamos el modelo FastText de Wikipedia en español que ya tenemos disponible localmente.

> **Para mirar:** ¿qué diferencia hay entre buscar "mate" con TF-IDF (donde aparece como columna con un número) y buscar "mate" en el espacio de embeddings (donde aparece como un vector de 300 números)?


In [1]:
import os
from gensim.models import FastText

# Configuración de ruta al modelo FastText local
ruta_fasttext = os.path.join(os.path.dirname(os.path.abspath("__file__")), "vectors", "wiki.es.bin")

# Si el path automático no funciona, usá una ruta absoluta:
# ruta_fasttext = r"E:\IFTS24-Cuadernos 2026\ifts24-lab-pln-2026\010_representaciones_semanticas_parte2\vectors\wiki.es.bin"

print(f"Ruta configurada: {ruta_fasttext}")
archivo_existe = os.path.exists(ruta_fasttext)
print(f"El archivo existe: {archivo_existe}")


Ruta configurada: c:\dominguez-micaela-belen-pln-1c-2026\010\002-PRA\vectors\wiki.es.bin
El archivo existe: True


### Cargando el modelo

La carga puede tardar varios minutos. Una vez en memoria, las consultas son instantáneas.


In [2]:
# Intentamos cargar el modelo FastText
modelo_cargado = False

try:
    print("Cargando el modelo FastText... (puede tardar varios minutos)")
    vectores_fasttext = FastText.load_fasttext_format(ruta_fasttext)
    total_palabras = len(vectores_fasttext.wv.index_to_key)
    print(f"Modelo cargado. Vocabulario: {total_palabras} palabras.")
    modelo_cargado = True

except FileNotFoundError:
    print(f"No se encontró el archivo: {ruta_fasttext}")
    print("Verificá que 'wiki.es.bin' esté en la carpeta 'vectors/'.")
    vectores_fasttext = None

except Exception as error:
    print(f"Error al cargar: {error}")
    vectores_fasttext = None


Cargando el modelo FastText... (puede tardar varios minutos)


C:\Users\domin\AppData\Local\Temp\ipykernel_16372\2050190581.py:6: DeprecationWarning: Call to deprecated `load_fasttext_format` (use load_facebook_vectors (to use pretrained embeddings) or load_facebook_model (to continue training with the loaded full model, more RAM) instead).
  vectores_fasttext = FastText.load_fasttext_format(ruta_fasttext)


Modelo cargado. Vocabulario: 985667 palabras.


### Obteniendo el vector de una palabra

En BoW/TF-IDF, "mate" era una columna con un número. En embeddings, "mate" es un vector de 300 números que codifica su significado semántico.

> **Para mirar:** ¿los primeros 10 valores del vector tienen algún patrón visible? ¿Podrías "leer" el significado directamente de esos números?


In [4]:
if modelo_cargado and vectores_fasttext is not None:
    # Obtenemos el vector de una palabra de nuestro corpus
    palabra_ejemplo = 'mate'

    # Verificamos que la palabra esté en el vocabulario
    if palabra_ejemplo in vectores_fasttext.wv:
        vector_mate = vectores_fasttext.wv[palabra_ejemplo]
        cantidad_dimensiones = len(vector_mate)
        print(f"Vector de '{palabra_ejemplo}':")
        print(f"  Dimensiones: {cantidad_dimensiones}")
        print(f"  Primeros 10 valores: {vector_mate[:10].round(4)}")
    else:
        # FastText puede construir vectores incluso para palabras OOV
        print(f"'{palabra_ejemplo}' no está en el vocabulario explícito.")
        print("FastText puede inferir un vector de todos modos usando n-gramas.")
else:
    print("El modelo no está cargado.")


Vector de 'mate':
  Dimensiones: 300
  Primeros 10 valores: [ 0.0553 -0.4486  0.3436 -0.2965  0.1276 -0.0546  0.0173 -0.4125 -0.1294
 -0.5784]


Y no porque solo son numeros decimales las 300 dimensiones son las que definen el espacio semantico al que pertenecen

### Similitudes semánticas con embeddings

A diferencia de TF-IDF, los embeddings permiten encontrar palabras semánticamente relacionadas aunque no compartan ninguna letra.

> **Para mirar:** ¿las palabras más similares a "mate" en el espacio de embeddings te parecen lógicas? ¿Aparecen palabras que no estarían relacionadas con TF-IDF?


In [5]:
if modelo_cargado and vectores_fasttext is not None:
    # Buscamos palabras similares a términos de nuestro corpus
    palabras_a_consultar = ['mate', 'amargo', 'caliente']

    for palabra in palabras_a_consultar:
        try:
            similares = vectores_fasttext.wv.most_similar(palabra, topn=5)
            print(f"Más similares a '{palabra}':")
            for palabra_similar, puntaje in similares:
                print(f"  {palabra_similar}: {puntaje:.4f}")
            print()
        except Exception as error:
            print(f"Error con '{palabra}': {error}")
else:
    print("El modelo no está cargado.")


Más similares a 'mate':
  jaque: 0.4637
  pavić: 0.4612
  yerba: 0.4544
  mates: 0.4393
  draganja: 0.4379

Más similares a 'amargo':
  amargor: 0.7758
  amargo»: 0.7754
  amargos: 0.7678
  amargoso: 0.7359
  amarg: 0.7111

Más similares a 'caliente':
  calienten: 0.8375
  calientes: 0.8374
  calienta: 0.8086
  calienes: 0.7949
  caliendo: 0.7692



yerba y mates si. Despues pavic jaque draganja son raras capaz confundio el juego de ajedrez. Si yerba con mate no tendrian relacion con tf idf

### Analogías vectoriales

Con embeddings podemos hacer álgebra semántica, algo imposible con BoW o TF-IDF.

> **Para mirar:** ¿la analogía tiene sentido? ¿Podés inventar una analogía relacionada con el vocabulario del corpus?


In [8]:
if modelo_cargado and vectores_fasttext is not None:
    # Analogía: casa - edificio + carpa = ?
    palabras_positivas = ['casa', 'edificio']
    palabras_negativas = ['carpa']

    try:
        resultado = vectores_fasttext.wv.most_similar(
            positive=palabras_positivas,
            negative=palabras_negativas,
            topn=3
        )
        print("Analogía: casa + edificio - carpa")
        print("Resultado más probable:")
        for palabra, puntaje in resultado:
            print(f"  {palabra}: {puntaje:.4f}")
    except Exception as error:
        print(f"Error en la analogía: {error}")
else:
    print("El modelo no está cargado.")


Analogía: casa + edificio - carpa
Resultado más probable:
  exedificio: 0.6232
  edifición: 0.6153
  edificio : 0.6049


Masomenos.


## 7. Comparación final: BoW vs. TF-IDF vs. Embeddings

| Característica | BoW | TF-IDF | Embeddings |
|---|---|---|---|
| **Tamaño del vector** | = tamaño del vocabulario (miles) | = tamaño del vocabulario (miles) | fijo y pequeño (ej: 300) |
| **¿Captura orden?** | No | No | Parcialmente (en sentence embeddings, sí) |
| **¿Captura semántica?** | No | Parcialmente | Sí |
| **¿Maneja OOV?** | No | No | FastText: sí; Word2Vec: no |
| **¿Permite analogías?** | No | No | Sí |
| **Costo computacional** | Bajo | Bajo | Medio-alto (carga del modelo) |
| **¿Cuándo usarlo?** | Clasificación simple, búsqueda por palabras exactas | Similar a BoW + mejora palabras clave | Similitud semántica, búsqueda semántica, clasificación avanzada |

> **Reflexión:** ¿para cuál de estas tareas usarías cada representación? (1) detectar spam, (2) recomendar artículos similares, (3) analizar sentimientos en reseñas.


Depende para detectar spam usaria tf idf. Para recomendar articulos embeddings y para analisis de sentimiento  embbedings. Pero todo depende del volumen o sea es muy generico para clasificar si para analizar sentimientos yo necesito de la semantica embeddigs pero para casos simples usaria tf idf 

## Guía de estudio — Síntesis e integración

## Preguntas y Respuestas Clave

### **Integración de Técnicas**

**P: ¿Cuál es la ventaja principal de usar Pipeline en scikit-learn?**  
R: Encadena automáticamente pasos (vectorización + clasificación) y evita data leakage aplicando transformaciones solo en datos de entrenamiento durante fit.

**P: ¿Por qué usar train_test_split antes de crear el pipeline?**  
R: Para evaluar objetivamente el rendimiento en datos no vistos y detectar overfitting del modelo completo.

**P: ¿Qué información proporciona classification_report?**  
R: Precision (de predicciones positivas, cuántas son correctas), Recall (de casos reales positivos, cuántos detectamos), y F1-score (balance entre ambas).

### **Comparación de Métodos**

**P: ¿Cuándo añadir n-gramas a TF-IDF?**  
R: Cuando el contexto local es importante ("muy bueno" vs "bueno muy"), pero aumenta dimensionalidad y complejidad.

**P: ¿Cómo comparar objetivamente BoW vs TF-IDF vs Embeddings?**  
R: Usar mismo pipeline, mismos datos de train/test, comparar métricas finales (F1-score, accuracy) en tarea específica.

**P: ¿Por qué la matriz de confusión es importante?**  
R: Muestra patrones de error específicos: ¿confunde más falsos positivos o falsos negativos? ¿Qué clases se confunden entre sí?

### **Embeddings en Pipelines**

**P: ¿Cómo integrar word embeddings en un pipeline de clasificación?**  
R: Promediando vectores de palabras del documento o usando representaciones más sofisticadas como weighted averages por TF-IDF.

**P: ¿Qué ventajas tienen embeddings sobre TF-IDF en clasificación?**  
R: Capturan semántica (sinónimos contribuyen similarmente), menor dimensionalidad, mejor generalización a vocabulario no visto.

### **Decisiones Prácticas**

**P: ¿Cómo decidir entre Naive Bayes y Logistic Regression?**  
R: NB asume independencia de features, rápido, bueno para texto. LR más flexible, maneja correlaciones, mejor con features continuas.

**P: ¿Cuándo usar cada método en producción?**  
R: BoW/TF-IDF para baseline rápido, FastText para robustez OOV, Word2Vec para análisis semántico, según requirements específicos.

### **Evaluación Integral**

**P: ¿Qué métricas usar para comparar sistemas de NLP?**  
R: Accuracy (básico), F1-score (balanceado), Precision/Recall específicos según costo de errores, tiempo de inferencia.

**P: ¿Cómo detectar overfitting en NLP?**  
R: Gran diferencia entre performance train/validation, alta dimensionalidad vs pocos datos, memorización de vocabulario específico.

## Puntos Clave para Recordar

1. **Pipeline automatiza y previene errores** en flujo ML
2. **Evaluación objetiva require train/test split** apropiado
3. **Diferentes métodos para diferentes problemas** - no hay silver bullet
4. **Métricas específicas revelan fortalezas/debilidades** de cada approach
5. **Embeddings aportan semántica** pero requieren más setup
6. **Integración práctica** considera tiempo, memoria, interpretabilidad

## Errores Comunes a Evitar

- Aplicar transformaciones a todo el dataset antes del split
- Comparar métodos con diferentes preprocessings
- Elegir solo por accuracy sin considerar precision/recall
- No validar assumptions de algoritmos (ej: independencia en NB)
- Ignorar computational requirements en producción

## Conexión con Próxima Clase

Esta integración de métodos tradicionales y modernos prepara para **aplicaciones avanzadas**: extracción de información, análisis de entidades, y sistemas de NLP end-to-end.

---
*Consejo: Siempre establece baseline simple (BoW + NB) antes de probar métodos complejos. Te da referencia para validar si la complejidad adicional vale la pena.*

## Referencias

- [Spanish Word Embeddings — dccuchile/spanish-word-embeddings](https://github.com/dccuchile/spanish-word-embeddings?tab=readme-ov-file)
- [Word vectors for 157 languages — FastText](https://fasttext.cc/docs/en/crawl-vectors.html)
